# Named Entity Recognition with Synthetic Data

<sup>*By [Bram Vanroy](https://www.linkedin.com/in/bramvanroy/) for [SSHOC-NL](https://sshoc.nl/)*</sup>

**Notebook 2: Generating the synthetic NER data**

In the previous notebook we trained an encoder model on an artificially small set of named entities. We found that its final performance was not great. Given the wide application potential of generative LLMs, we therefore explore whether we can generate synthetic data with LLMs, and mix it in our training set to improve downstream performance on a new encoder model. In other words, we are *augmenting* our training set with subpar quality data (annotations will not be perfect) in hopes to improve the generalizability of our trained model.

✅  To be manageable in the scope of a workshop *and* to run on Google Colab, some concessions were made here: we are only generating a small dataset with a (relatively) small LLM. However, if you have the time and hardware to explore further, I highly encourage you to create more data and/or use a larger LLM, and see how that (undoubtedly positively) impacts your final performance.

First, let's get some questions out of the way.

- Can't I just use an LLM to do the classification and cut out this whole middle process? What's the point?
> Yes you can! But there are cases where you may not want to, for instance: if you will have lots of data to run through your pipeline, it is cheaper (time/money/compute/environment-wise) to generate synthetic training data once and train a smaller classification model with that data compared to always use an LLM. Secondly, and more importantly, this notebook is intended to show what is methodologically possible and sound. How to train your own model, how to generate synthetic data based on in-domain data, and how to evaluate and compare models.
- So doesn't that mean we are effectively "distilling" the capabilities of the large LLM into a smaller model?
> Exactly, this is a form of distillation! By generating synthetic data with a large model, and optimizing (finetuning) a smaller model, we are teaching the smaller model to mimic the larger model. That way we end up with a small, efficient, and reusable model. This only works, of course, if the larger model can produce good quality data that our smaller model can learn from.
- Is an LLM the best way to label named entities in existing data? That seems very expensive when we might just use existing recognizers built into [spaCy](https://spacy.io/usage/linguistic-features#named-entities), [stanza](https://stanfordnlp.github.io/stanza/ner.html), [GLiNER](https://github.com/urchade/GLiNER), and so on.
> True! This notebook should be considered for its methodological inspiration. If you are interested exclusively in high-performance NER systems, then this notebook is **not** for you.

Legend

- ⚠️ => warning/strong recommendation. If not followed, it might lead to your code not working!
- ✅ => to-do recommendation/try it yourself
- 💡 => insight

---

Now, let's prepare our environment. It is tailored to run on Google Colab with a GPU enabled, which is a specialized piece of hardware that is much more efficient than a regular CPU when it comes to using neural networks.

> ⚠️ **To select a free GPU, go to "Runtime > Change runtime type" and select a GPU**!


In [1]:
!uv pip install --upgrade -q vllm==0.11.0 transformers==4.57.1 triton==3.4 kernels torch==2.8.0
!uv pip install --upgrade -q numpy==2.0.2
!uv pip uninstall -q torchvision torchaudio

In [2]:
import locale
locale.getpreferredencoding = lambda: "UTF-8"

In [3]:
import torch

# Check if CUDA is available
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    total_mem_gb = gpu.total_memory / (1024 ** 3)
    print(f"GPU: {gpu.name}")
    print(f"Total memory: {total_mem_gb:.2f} GB")
else:
    raise ValueError("No GPU detected! Did you click 'Runtime > Change Runtime' and select a GPU?")

GPU: NVIDIA L40
Total memory: 44.39 GB


⚠️ This notebook relies heavily on the free Hugging Face infrastructure to upload/download our models and data. So before getting started you should create a Hugging Face (HF) account. Do not have an account yet? [Register now!](https://huggingface.co/join)

In the HF settings, go to Settings and find Access tokens (or go [here](https://huggingface.co/settings/tokens)). Create a new token. Under "User permissions" select all options under Repositories, or (less secure) simply select "Write". Scroll down and click "Create token". **Immediately copy this token!** If you close the modal, the Secret key will not be visible anymore for security reasons.

**Google Colab:** Once you have copied the code, go back to Colab (or this notebook), and on the left sidebar click on the key icon ("Secrets"). As a name, use `HF_TOKEN` (exactly!) and as a value, paste your copied Secret. In the future, and with this notebook if you are planning to run it, enable "Notebook access", indicating that this specific notebook can access that Secret key. Other people will never see your key.

**Locally:** Using the secret key/token that you retrieved above you can also log in on your own device. You can do that with `hf auth login` on the command line, following [this guide](https://huggingface.co/docs/huggingface_hub/en/guides/cli#hf-auth-login).

⚠️ **After entering the Secret in Colab or logging in on your own PC, you may need to restart the session (Runtime > Restart session).**

In [4]:
from huggingface_hub import whoami
from huggingface_hub.errors import LocalTokenNotFoundError

# Set to False if you do not want to push models to the Hub
DO_USE_HUB = True  

try:
    whoami = whoami()
except LocalTokenNotFoundError:
    HF_ACCOUNT = None
    print("⚠️ No Hugging Face username found. If you want to run this notebook with all functionalities, you need a logged in account. Follow the steps above.")
else:
    if whoami and "name" in whoami and whoami["type"] == "user":
        HF_ACCOUNT = whoami["name"]
        print(f"Logged in as {HF_ACCOUNT}!")
    else:
        HF_ACCOUNT = None
        print("⚠️ No Hugging Face username found. If you want to run this notebook with all functionalities, you need a logged in account. Follow the steps above.")

Logged in as BramVanroy!


## Loading the LLM

Generative large language models have proven their applicability in the last years. Unlike our specialized, small-and-optimized token classifcation model that we trained in the previous notebook, these LLMs can be prompted to perform a wide variety of tasks. In a sense that also makes them less efficient for specific tasks: a single, specialized model often matches or outperforms generative large models at a fraction of the cost (lower carbon footprint, faster, lower hardware requirements). Assuming that we have large corpora to process with a NER model, it is not feasible to run a generative LLM at that scale. But what we can do, is improve our small token classification model with "synthetic data", data that is fully created by a generative model. We are using the LLM as a facilitator, as a data generator. In a sense, this is distillation: we are traing our small model on outputs of a larger "teacher" model, with the notable exception that we are also including human-annotated "gold" data.

In this notebook we are using [vLLM](https://docs.vllm.ai/en/latest/), an LLM framework optimized for inference (not training). It is relatively easy to use, although setting it up for maximal speed may be a hassle depending on your hardware -- do not worry, the default settings given here should just work on Colab. vLLM is compatible with popular models on the [Hugging Face hub](https://huggingface.co/models), which makes our lives easier.

The choice of language model is tough. First of all there are [many to choose from](https://huggingface.co/models) and their model cards do not always provide a clear overview of their benchmark performance! The model choice in this case ([Qwen3 14B](https://huggingface.co/Qwen/Qwen3-14B-AWQ)) is based on my prior positive experience with the model and the tough constraint of working (well) on Google Colab while providing maximal performance. It is a generative decoder-only model that punches above its weight by competing with (and often out-performing) larger models like Gemma 3 27B on English benchmarks ([paper](https://arxiv.org/abs/2505.09388)).

✅ Depending on your available compute budget, you may first want to evaluate a number of generative LLMs on the NER validation set that we use. Then use the best model to generate synthetic data in this notebook. 

First we will (down)load the model. Loading LLMs with `vLLM` is different from what you may be used to with `transformers`. We specify in advance a number of parameters such as the batch size and maximum model length, which allows vLLM to pre-optimize the model for our type of workload. You can find more information [here](https://docs.vllm.ai/en/v0.8.3/serving/engine_args.html) and tailor it to custom hardware to maximize the speed (throughput) but that is out of scope for this notebook. The settings here suffice to work on Google Colab. We are using a quantized version of the model -- [quantization](https://huggingface.co/docs/transformers/v4.57.1/quantization/overview) is a technique (or well, multiple techniques) that reduces the memory usage of a model at a cost of lower precision. So it enables you to run larger models at a small cost in performance.

- ⚠️ Executing the cell below will download the Qwen model (~10GB). This may take a long time (~10-15 minutes). Do not restart your notebook or click on any other cells; be patient!
- ⚠️ You can ignore error `AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'` errors.
- ⚠️ If you get an error about numpy core, restart and re-run the notebook. (Runtime > Restart session and run all) It means that the libraries that we installed are not loaded correctly yet.

In [5]:
from transformers import set_seed
from vllm import LLM, SamplingParams

# ⚠️⚠️⚠️
# THIS CELL MAY TAKE A LONG TIME TO RUN WITHOUT ANY OUTPUT (~10-15 minutes)
# DO NOT RESTART THE NOTEBOOK -- BE PATIENT -- the model is downloading
# You can ignore error `AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'`
# ⚠️⚠️⚠️

set_seed(42)

MODEL_NAME = "Qwen/Qwen3-14B-AWQ"
# Always look for official sampling parameters, e.g.
# on the model card https://huggingface.co/Qwen/Qwen3-14B-AWQ
SAMPLING_PARAMETERS = SamplingParams(
    temperature=0.7,
    top_p=0.8,
    top_k=20,
    min_p=0.0,
    max_tokens=1024
)
# Disable "thinking" animation for faster generation - it is not super useful for us
# Note that this is different between models and may not be available
# for your model, see https://discuss.vllm.ai/t/how-to-disable-thinking-for-different-model/1499
CHAT_TEMPLATE_KWARGS = {
    "enable_thinking": False
}

# If you are running locally with more VRAM you may increase the batch size and max model len
BATCH_SIZE = 4
MAX_MODEL_LEN = 2048
model = LLM(
    model=MODEL_NAME,
    max_model_len=MAX_MODEL_LEN,
    max_num_seqs=BATCH_SIZE,
    gpu_memory_utilization=0.95
)

`torch_dtype` is deprecated! Use `dtype` instead!


[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0


(EngineCore_DP0 pid=124823) FlashInfer is not available. Falling back to the PyTorch-native implementation of top-p & top-k sampling. For the best performance, please install FlashInfer.
Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  50% Completed | 1/2 [00:01<00:01,  1.35s/it]
Loading safetensors checkpoint shards: 100% Completed | 2/2 [00:02<00:00,  1.43s/it]
Loading safetensors checkpoint shards: 100% Completed | 2/2 [00:02<00:00,  1.42s/it]
(EngineCore_DP0 pid=124823) 
Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 4/4 [00:00<00:00, 38.03it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 3/3 [00:00<00:00, 39.08it/s]


## Preparing the prompt

Just as you're used to from using tools like ChatGPT, Gemini, Claude, Copilot and so on, we will need to write a good prompt to get sensible results. When using local models, you sometimes have to make trade-offs between very long, extensive prompts and speed and memory/computational power. Running longer texts requires more memory and is slower to process! So I have made a few attempts that yield decent results. If you have more computational power available, you may try to write a prompt based on (or including) the [official UNER guidelines](https://www.universalner.org/guidelines/).

I have also added five examples, taken from the UNER validation set, to show the type of data structure that is expected. The model should then follow the same structure, which makes it easy for us to write a function that validates the output data and only extracts data samples that are in the format as we expect. The validation function is relatively simple: it will only extract data samples that use `O`, `LOC`, `PER`, `ORG` labels (with B- or I-) and only return items where the number of tokens and labels is identical. Finally we also verify the IOB2 schema: an `I-*` tag should always be preceded by a `B-*` or `I-*` tag of the same type.

In [6]:
import re
from typing import Generator


example_data = """\
# Crude-oil prices rose Wednesday as strengthening Hurricane Rita, now a Category 5 storm, threatened to disrupt oil production in the Gulf of Mexico.
Crude	O
-	O
oil	O
prices	O
rose	O
Wednesday	O
as	O
strengthening	O
Hurricane	O
Rita	O
,	O
now	O
a	O
Category	O
5	O
storm	O
,	O
threatened	O
to	O
disrupt	O
oil	O
production	O
in	O
the	O
Gulf	B-LOC
of	I-LOC
Mexico	I-LOC
.	O

# There 's also a Miramar in California, the site of a rather large Air Force Base...
There	O
's	O
also	O
a	O
Miramar	B-LOC
in	O
California	B-LOC
,	O
the	O
site	O
of	O
a	O
rather	O
large	O
Air	B-LOC
Force	I-LOC
Base	I-LOC
...	O

# Marlene Hilliard
Marlene	B-PER
Hilliard	I-PER

# i 'm doing a report on how afghanistan and Vietam are different and alike.
i	O
'm	O
doing	O
a	O
report	O
on	O
how	O
afghanistan	B-LOC
and	O
Vietam	B-LOC
are	O
different	O
and	O
alike	O
.	O

# Once upon a time (in 2001, to be specific), the Coca-Cola corporation built a bottling plant in a small and remote Indian village in the state of Kerala.
Once	O
upon	O
a	O
time	O
(	O
in	O
2001	O
,	O
to	O
be	O
specific	O
)	O
,	O
the	O
Coca	B-ORG
-	I-ORG
Cola	I-ORG
corporation	O
built	O
a	O
bottling	O
plant	O
in	O
a	O
small	O
and	O
remote	O
Indian	O
village	O
in	O
the	O
state	O
of	O
Kerala	B-LOC
.	O

# As such, it is essential that HANO comply with 2003 enforcement agreement," said James Perry, GNOFHAC Executive Director
As	O
such	O
,	O
it	O
is	O
essential	O
that	O
HANO	B-ORG
comply	O
with	O
2003	O
enforcement	O
agreement	O
,	O
"	O
said	O
James	B-PER
Perry	I-PER
,	O
GNOFHAC	B-ORG
Executive	O
Director	O

"""

# This prompt could be improved further by incorporating the UNER annotation guidelines
# but we want to keep context length low for cost and speed reasons.
# https://www.universalner.org/guidelines/
base_messages = [{
    "role": "user",
    "content": f"""**Task:**
Generate **20 sentences** annotated for Named Entity Recognition (NER) in IOB2 format.

**Entity types allowed:**

* PER: person names (real or fictional person names)
* ORG: organizations (any named collection of people, such as firms, institutions, organizations, artists, sports teams, political parties etc.)
* LOC: locations (airports, restaurants, hotels, tourist attractions, shops, street addresses, oceans, fjords, planets, parks and fictional locations)

**Important annotation rules:**

* Tag only **named entities** that meet the UNER criteria: they are proper nouns or include proper nouns; they refer to a unique entity with a constant reference.
* Use **B-TYPE** to mark the **first token** of an entity span, and **I-TYPE** for subsequent tokens of the same span; all other tokens get **O**.
* Strictly follow IOB2 format: I-TYPE tags should always be preceded by a B-TYPE or I-TYPE of the same type to constitute a single entity span.
* Entities must be **flat spans**, never nested. E.g., annotate [University of Washington St. Louis](ORG), not [University of Washington](LOC) [St. Louis](LOC).
* In case of ambiguity (e.g., same string could be ORG or LOC), choose the **literal meaning** or the **most common usage** given the context.
* Do *not* annotate:
  * Time expressions
  * Nationalities or languages or adjectives derived from them
* For LOC: include geographical places, buildings, facilities, street addresses, etc. E.g., [The Gulf of Mexico](LOC)
* For ORG: include named collections of people, institutions, sports teams, etc. Corporate designators (Co., Ltd.) are part of the ORG name.
* For PER: include individual people (real or fictional), including names with initials, nicknames, etc. E.g., [Mr. Grinch](PER), [Charlie Chaplin](PER).

**Output format:**
For each sentence:

1. Start with a line beginning with `# ` followed by the sentence.
2. On the following lines, list **each token** of the sentence followed by a tab character `\t` then the IOB2 tag. One token per line.

**Additional guidance:**

* Include **variation** in topics (blogs, newsgroups, emails, reviews, Q&A), sentence length, syntax, and word order.
* Use both **single-token entities** (just B-TYPE) and **multi-token entities** (B-TYPE + I-TYPE).
* Do *not* include any extra explanation, only the annotated examples.

**Examples:**
{example_data}
"""
}]

NER_TAGS = ["B-PER", "I-PER", "B-ORG", "I-ORG", "B-LOC", "I-LOC", "O"]
NER_RE = re.compile(r"^(\S+)\s+(" + "|".join(NER_TAGS) + r")$")

def is_valid_iob2(tags: list[str]) -> bool:
    """Check whether a list of tags is valid IOB2 format so that 
    "I-TYPE" tags are only preceded by "B-TYPE" or "I-TYPE" of the same type.
    """
    prev_prefix = "O"
    prev_type = ""
    for tag in tags:
        if tag == "O":
            prev_prefix = "O"
            prev_type = ""
            continue
        if "-" not in tag:
            return False
        prefix, ent_type = tag.split("-", 1)
        if prefix == "B":
            prev_prefix = "B"
            prev_type = ent_type
        elif prefix == "I":
            # "I" must follow "B" or "I" of the same type
            if prev_prefix not in ("B", "I") or prev_type != ent_type:
                return False
            prev_prefix = "I"
        else:
            return False
    return True

def extract_annotations(text: str) -> Generator[dict, None, None]:
    """Extract NER annotations from generated text that follow the structure
    as specified in `example_data` above. Each yielded item is a dict with keys:
    - "text": the original sentence
    - "tokens": list of tokens
    - "ner_tags": list of NER tags corresponding to the tokens
    """
    text = text.strip()

    # Extract blocks
    blocks = []
    block = []
    for line in text.split("\n"):
        line = line.strip()
        if line.startswith("#") or line == "":
            if block:
                blocks.append(block)
                block = []
            if line:
                block.append(line)
        else:
            block.append(line)
    if block:
        blocks.append(block)

    # Validate and filter blocks
    for block in blocks:
        sentence = block[0].lstrip("#").strip()
        try:
            tokens, tags = zip(*[NER_RE.match(line).groups() for line in block[1:] if NER_RE.match(line)])
        except Exception:
            continue

        tags = [t.strip() for t in tags]
        tokens = [t.strip() for t in tokens]
        if any(not tag for tag in tags) or any(not token for token in tokens) or len(tokens) != len(tags) or len(tokens) == 0:
            continue
        
        low_sent = "".join(sentence.lower().split())
        low_toks = "".join(tokens).lower()
        if low_sent != low_toks:
            continue
        
        if not is_valid_iob2(tags):
            continue
        
        # Only finally yield (return) and item if all checks passed
        yield {"tokens": tokens, "ner_tags": tags}


## Generate the data!

With everything set in place, we can now start generating our data samples! We basically keep feeding the same prompt over and over again to the model and the model will always keep generating new data. The model will generate more than one sentence per output, so we keep track of the actual number of generated lines and stop generating as soon as we have reached it. As a safe-guard we of course check the validity of the data with the function described above and we also explicitly check for (and discard) duplicates.

- ✅ In reality you would first generate a handful of sentences, verify the quality and output format, and iterate on the prompt to improve it further. Feel free to do so!
- ✅ To add more variation, you could also randomly select the five examples from your validation set (or a hand-written pool of examples) and add those as few-shot examples in your prompt.

Note: Generating 200 samples will take around 1-1.5 hours to run on Google Colab. For reference, generating 2000 samples (x10) on an L40S GPU takes ~20 minutes.

In [7]:
from datasets import Dataset
from transformers import set_seed
from tqdm import tqdm

set_seed(42)

# Target dataset size
NUM_TARGET_SENTENCES = 200

data = []
sentences = set()
pbar = tqdm(total=NUM_TARGET_SENTENCES, desc="Generating synthetic NER data")
while len(data) < NUM_TARGET_SENTENCES:
    # Prepare batch of messages
    batch_size = min(BATCH_SIZE, NUM_TARGET_SENTENCES - len(data))
    batch_messages = [base_messages] * batch_size

    # Generate responses
    outputs = model.chat(
        batch_messages,
        SAMPLING_PARAMETERS,
        use_tqdm=False,
        chat_template_kwargs=CHAT_TEMPLATE_KWARGS
    )
    for output in outputs:
        text = output.outputs[0].text
        for annotation in extract_annotations(text):
            text = "".join(annotation["tokens"]).lower()
            if text in sentences:
                continue

            sentences.add(text)
            data.append(annotation)
            pbar.update(1)
            pbar.refresh()

            if len(data) >= NUM_TARGET_SENTENCES:
                break
        if len(data) >= NUM_TARGET_SENTENCES:
            break
pbar.close()

synth_ds = Dataset.from_list(data)
print(synth_ds)

Generating synthetic NER data: 100%|██████████| 200/200 [01:36<00:00,  2.07it/s]

Dataset({
    features: ['tokens', 'ner_tags'],
    num_rows: 200
})


In [8]:
for item in synth_ds.shuffle(64).take(5):
    for token, tag in zip(item['tokens'], item['ner_tags']):
        print(f"{token}\t{tag}")
    print()

The	O
new	O
president	O
of	O
the	O
company	O
is	O
Maria	B-PER
Lopez	I-PER
.	O

The	O
new	O
restaurant	O
,	O
called	O
The	O
Spice	O
Garden	O
,	O
opened	O
last	O
week	O
in	O
downtown	O
Seattle	B-LOC
.	O

The	O
company	O
Tesla	B-ORG
was	O
founded	O
by	O
Elon	B-PER
Musk	I-PER
,	O
a	O
well-known	O
entrepreneur	O
.	O

Amazon	O
Prime	B-ORG
Day	I-ORG
is	O
an	O
annual	O
event	O
hosted	O
by	O
Amazon	B-ORG
.	I-ORG
com	O
,	O
Inc	O
.	O

The	O
Louvre	B-LOC
Museum	I-LOC
in	O
Paris	B-LOC
houses	O
the	O
Mona	O
Lisa	O
,	O
a	O
painting	O
by	O
Leonardo	B-PER
da	I-PER
Vinci	I-PER
.	O



Remember, we cannot expect "perfect" data, especially given the relatively small model and the compressed annotation guidelines in the prompt! The goal of generating synthetic data is to provide at least some positive signal during training that can push the model in the right direction.

Before saving and uploading our datasets, we simply have to make sure the labels are correctly "cast" to the same features that are used in the UNER dataset. If you remember from the previous notebook, the features provide shortcuts such as converting labels to indices (e.g. `"O": 0`). So of course we want to mapping between labels and indices to be identical between the datasets that we are about to merge! So in the cell below we re-use the features in the exact order as the UNER dataset.

In [9]:
from datasets import load_dataset, List, ClassLabel

uner_ds = load_dataset("BramVanroy/universal_ner", "en_ewt", split="train")
ner_feats = uner_ds.features["ner_tags"].feature.names

print(f"Labels of the original dataset: {ner_feats}")
label_feats = ClassLabel(names=ner_feats)
# The ner_tags column needs to be cast to the correct ClassLabel feature
# Note that it is a `List` of `ClassLabel`s (one for each token)
synth_ds = synth_ds.cast_column("ner_tags", List(label_feats))

Labels of the original dataset: ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC']


Casting the dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

With that done, we can upload the data to the Hugging Face hub! That way it is easy to integrate this new dataset in other scripts, as we will soon see. By default, in the spirit of openness, uploading data and models are open to the public. However, if you feel uncomfortable with that, you can also make them private so only you have access to it.

In [10]:
PRIVATE = False

ds_name = f"synthetic-uner-ner-{NUM_TARGET_SENTENCES}-{MODEL_NAME.split('/')[-1]}"
synth_ds.save_to_disk(ds_name)

if HF_ACCOUNT is not None:
    synth_ds.push_to_hub(f"{HF_ACCOUNT}/{ds_name}", private=PRIVATE)
    print(f"Dataset pushed to the Hub at https://hf.co/datasets/{HF_ACCOUNT}/{ds_name}!")
    print(f"Data hub ID to copy: {HF_ACCOUNT}/{ds_name}")
else:
    print(f"Dataset directory to copy: {ds_name}")

Saving the dataset (0/1 shards):   0%|          | 0/200 [00:00<?, ? examples/s]

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md:   0%|          | 0.00/474 [00:00<?, ?B/s]

Dataset pushed to the Hub at https://hf.co/datasets/BramVanroy/synthetic-uner-ner-200-Qwen3-14B-AWQ!
Data hub ID to copy: BramVanroy/synthetic-uner-ner-200-Qwen3-14B-AWQ


Back to the other notebook!